# MotorNet-JAX: States and Simulation

This notebook covers the state representation in MotorNet-JAX and how to run simulations efficiently.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('.')))))

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

import motornet_jax as mn
from motornet_jax.types import JointState, MuscleState, GeometryState, EffectorState, CartesianState
from motornet_jax.effector import Arm26

print('All packages imported.')
print(f'JAX version: {jax.__version__}')

## I. State Types in MotorNet-JAX

MotorNet-JAX uses **NamedTuples** for state representation. These are:
- Immutable (required for JAX)
- PyTree-compatible (work with jax.jit, jax.vmap, jax.grad)
- Self-documenting (named fields)

### State Hierarchy:

```
EffectorState
├── joint: JointState
│   ├── position (batch, n_joints)
│   └── velocity (batch, n_joints)
├── cartesian: CartesianState
│   ├── position (batch, n_dim)
│   └── velocity (batch, n_dim)
├── muscle: MuscleState
│   ├── activation (batch, n_muscles)
│   ├── fiber_length (batch, n_muscles)
│   └── fiber_velocity (batch, n_muscles)
├── geometry: GeometryState
│   ├── musculotendon_length (batch, n_muscles)
│   ├── musculotendon_velocity (batch, n_muscles)
│   └── moment_arm (batch, n_muscles, n_joints)
└── fingertip (batch, n_dim)
```

In [ ]:
# Create an Arm26 and inspect its state
arm = Arm26(dt=0.01)
params = arm.get_params()

# Reset to get initial state
batch_size = 4
key = jax.random.PRNGKey(0)
state = arm.reset(batch_size=batch_size, key=key)

print("EffectorState fields:")
for field in state._fields:
    value = getattr(state, field)
    if hasattr(value, 'shape'):
        print(f"  {field}: shape {value.shape}")
    elif hasattr(value, '_fields'):
        print(f"  {field}: {type(value).__name__}")
        for subfield in value._fields:
            subvalue = getattr(value, subfield)
            print(f"    .{subfield}: shape {subvalue.shape}")

## II. Working with States

Since states are immutable NamedTuples, you create new states using `_replace()`:

In [ ]:
# Access state values
print("Joint positions:")
print(state.joint.position)

print("\nMuscle activations:")
print(state.muscle.activation)

print("\nFingertip positions:")
print(state.fingertip)

In [ ]:
# Modify state using _replace
# Set all activations to 0.5
new_activation = jnp.ones((batch_size, 6)) * 0.5
new_muscle_state = state.muscle._replace(activation=new_activation)
new_state = state._replace(muscle=new_muscle_state)

print("Original activation:")
print(state.muscle.activation)

print("\nNew activation:")
print(new_state.muscle.activation)

## III. Initializing States

You can initialize the arm with:
1. Random positions (default)
2. Specific joint angles

In [ ]:
# Random initialization
key = jax.random.PRNGKey(42)
random_state = arm.reset(batch_size=3, key=key)
print("Random joint positions:")
print(random_state.joint.position)

# Specific initialization
specific_joint = JointState(
    position=jnp.array([[0.5, 1.0], [1.0, 1.5], [1.5, 2.0]]),
    velocity=jnp.zeros((3, 2)),
)
specific_state = arm.reset(batch_size=3, joint_state=specific_joint)
print("\nSpecific joint positions:")
print(specific_state.joint.position)

## IV. Running Simulations

### Single Step
Use `Arm26.step()` for a single timestep:

In [ ]:
# Single step
action = jnp.ones((batch_size, 6)) * 0.3  # Muscle excitation
endpoint_load = jnp.zeros((batch_size, 2))  # External force on hand
joint_load = jnp.zeros((batch_size, 2))  # External torque on joints

state = arm.reset(batch_size=batch_size, key=key)
new_state = Arm26.step(state, action, endpoint_load, joint_load, params)

print("Position change after one step:")
print(new_state.joint.position - state.joint.position)

### Multi-Step with lax.scan

For efficient trajectory simulation, use `jax.lax.scan`:

In [ ]:
# Efficient multi-step simulation
@partial(jax.jit, static_argnums=(4,))
def simulate(state, action, endpoint_load, joint_load, n_steps):
    """Run simulation for n_steps, recording trajectory."""
    def step_fn(state, _):
        new_state = Arm26.step(state, action, endpoint_load, joint_load, params)
        # Record what we want
        output = {
            'fingertip': new_state.fingertip,
            'joint_pos': new_state.joint.position,
            'activation': new_state.muscle.activation,
        }
        return new_state, output
    
    final_state, trajectory = jax.lax.scan(step_fn, state, None, length=n_steps)
    return final_state, trajectory

# Run simulation
n_steps = 100
state = arm.reset(batch_size=batch_size, key=key)
action = jnp.ones((batch_size, 6)) * 0.5

final_state, trajectory = simulate(state, action, endpoint_load, joint_load, n_steps)

print(f"Trajectory shapes:")
for key, value in trajectory.items():
    print(f"  {key}: {value.shape}")

In [ ]:
# Visualize trajectory
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
t = np.arange(n_steps) * params.dt

# Fingertip X position
ax = axes[0]
for i in range(batch_size):
    ax.plot(t, trajectory['fingertip'][:, i, 0], label=f'Batch {i}')
ax.set_xlabel('Time (s)')
ax.set_ylabel('X position (m)')
ax.set_title('Fingertip X Position')
ax.legend()
ax.grid(True, alpha=0.3)

# Joint angles
ax = axes[1]
for i in range(batch_size):
    ax.plot(t, np.rad2deg(trajectory['joint_pos'][:, i, 0]), 
            label=f'Shoulder {i}', linestyle='-')
    ax.plot(t, np.rad2deg(trajectory['joint_pos'][:, i, 1]), 
            label=f'Elbow {i}', linestyle='--')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (deg)')
ax.set_title('Joint Angles')
ax.grid(True, alpha=0.3)

# Muscle activation (first batch only)
ax = axes[2]
for m in range(6):
    ax.plot(t, trajectory['activation'][:, 0, m], label=arm.muscle_names[m])
ax.set_xlabel('Time (s)')
ax.set_ylabel('Activation')
ax.set_title('Muscle Activation (Batch 0)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## V. Time-Varying Actions

To simulate with time-varying muscle commands, pass actions through scan:

In [ ]:
@partial(jax.jit, static_argnums=(4,))
def simulate_with_actions(state, actions, endpoint_load, joint_load, n_steps):
    """Simulate with time-varying actions."""
    def step_fn(state, action):
        new_state = Arm26.step(state, action, endpoint_load, joint_load, params)
        return new_state, new_state.fingertip
    
    final_state, trajectory = jax.lax.scan(step_fn, state, actions)
    return final_state, trajectory

# Create time-varying action: pulse to different muscles
n_steps = 300
actions = jnp.zeros((n_steps, 1, 6))  # (time, batch, muscles)

# Pulse pectoralis from t=50 to t=100
actions = actions.at[50:100, :, 0].set(1.0)
# Pulse deltoid from t=150 to t=200
actions = actions.at[150:200, :, 1].set(1.0)
# Pulse biceps from t=250 to t=300
actions = actions.at[250:300, :, 4].set(1.0)

state = arm.reset(batch_size=1, joint_state=JointState(
    position=jnp.array([[1.0, 1.5]]),
    velocity=jnp.zeros((1, 2)),
))

final_state, trajectory = simulate_with_actions(
    state, actions, jnp.zeros((1, 2)), jnp.zeros((1, 2)), n_steps
)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
t = np.arange(n_steps) * params.dt

# Actions
ax = axes[0]
for m in [0, 1, 4]:  # Pectoralis, Deltoid, Biceps
    ax.fill_between(t, 0, actions[:, 0, m], alpha=0.3, label=arm.muscle_names[m])
ax.set_ylabel('Action (excitation)')
ax.set_title('Muscle Commands')
ax.legend()
ax.grid(True, alpha=0.3)

# Trajectory
ax = axes[1]
ax.plot(t, trajectory[:, 0, 0], label='X')
ax.plot(t, trajectory[:, 0, 1], label='Y')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Position (m)')
ax.set_title('Hand Position')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## VI. Batched Simulation with Different Actions

Use `vmap` to simulate multiple trajectories with different actions in parallel:

In [ ]:
# Create different constant actions for each batch element
n_conditions = 8
n_steps = 200

# Different activation levels for the first muscle
activation_levels = jnp.linspace(0.0, 1.0, n_conditions)
actions = jnp.zeros((n_conditions, 6))
actions = actions.at[:, 0].set(activation_levels)  # Pectoralis

# Same initial state for all
initial_joint = JointState(
    position=jnp.tile(jnp.array([[1.0, 1.2]]), (n_conditions, 1)),
    velocity=jnp.zeros((n_conditions, 2)),
)
state = arm.reset(batch_size=n_conditions, joint_state=initial_joint)

# Simulate
final_state, trajectory = simulate(
    state, actions, 
    jnp.zeros((n_conditions, 2)), 
    jnp.zeros((n_conditions, 2)), 
    n_steps
)

# Plot trajectories
plt.figure(figsize=(10, 8))

colors = plt.cm.viridis(np.linspace(0, 1, n_conditions))
for i in range(n_conditions):
    plt.plot(trajectory['fingertip'][:, i, 0], trajectory['fingertip'][:, i, 1], 
             color=colors[i], linewidth=2, 
             label=f'Pectoralis = {activation_levels[i]:.1f}')
    plt.scatter([trajectory['fingertip'][-1, i, 0]], 
                [trajectory['fingertip'][-1, i, 1]], 
                color=colors[i], s=50, marker='x')

plt.scatter([trajectory['fingertip'][0, 0, 0]], 
            [trajectory['fingertip'][0, 0, 1]], 
            color='red', s=100, marker='*', zorder=10, label='Start')

plt.xlabel('X position (m)')
plt.ylabel('Y position (m)')
plt.title('Effect of Pectoralis Activation Level on Hand Trajectory')
plt.legend(loc='upper left')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## VII. External Loads

You can apply external forces to simulate:
- Gravity effects
- Force perturbations
- Object manipulation

In [ ]:
# Simulate with different endpoint loads
n_steps = 200
batch_size = 4

# Different loads for each batch
endpoint_loads = jnp.array([
    [0.0, 0.0],    # No load
    [5.0, 0.0],    # Rightward force
    [-5.0, 0.0],   # Leftward force
    [0.0, -5.0],   # Downward force
])

initial_joint = JointState(
    position=jnp.tile(jnp.array([[1.0, 1.2]]), (batch_size, 1)),
    velocity=jnp.zeros((batch_size, 2)),
)
state = arm.reset(batch_size=batch_size, joint_state=initial_joint)

# No muscle activation - just passive response to load
action = jnp.zeros((batch_size, 6))

final_state, trajectory = simulate(
    state, action, endpoint_loads, jnp.zeros((batch_size, 2)), n_steps
)

# Plot
plt.figure(figsize=(10, 8))

labels = ['No load', 'Rightward (5N)', 'Leftward (5N)', 'Downward (5N)']
colors = ['black', 'red', 'blue', 'green']

for i, (label, color) in enumerate(zip(labels, colors)):
    plt.plot(trajectory['fingertip'][:, i, 0], trajectory['fingertip'][:, i, 1], 
             color=color, linewidth=2, label=label)
    
    # Draw load arrow
    start = trajectory['fingertip'][0, i]
    plt.annotate('', xy=start + endpoint_loads[i]*0.02, xytext=start,
                 arrowprops=dict(arrowstyle='->', color=color, lw=2))

plt.scatter([trajectory['fingertip'][0, 0, 0]], 
            [trajectory['fingertip'][0, 0, 1]], 
            color='orange', s=100, marker='*', zorder=10, label='Start')

plt.xlabel('X position (m)')
plt.ylabel('Y position (m)')
plt.title('Arm Response to External Loads (No Muscle Activation)')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## VIII. Gradients Through Simulation

One of JAX's key features is automatic differentiation through simulations:

In [ ]:
# Compute gradient of final position with respect to action
def final_position_x(action):
    """Return final X position given constant action."""
    action_batched = action[None, :]  # Add batch dimension
    state = arm.reset(batch_size=1, joint_state=JointState(
        position=jnp.array([[1.0, 1.2]]),
        velocity=jnp.zeros((1, 2)),
    ))
    
    def step_fn(state, _):
        new_state = Arm26.step(state, action_batched, 
                               jnp.zeros((1, 2)), jnp.zeros((1, 2)), params)
        return new_state, None
    
    final_state, _ = jax.lax.scan(step_fn, state, None, length=100)
    return final_state.fingertip[0, 0]  # X position

# Compute gradient
grad_fn = jax.grad(final_position_x)

action = jnp.array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5])
gradient = grad_fn(action)

print("Gradient of final X position w.r.t. each muscle:")
for i, name in enumerate(arm.muscle_names):
    print(f"  {name}: {gradient[i]:.4f}")

In [ ]:
# Visualize gradient as sensitivity
plt.figure(figsize=(10, 5))

colors = ['blue' if g > 0 else 'red' for g in gradient]
plt.bar(arm.muscle_names, gradient, color=colors)
plt.axhline(0, color='black', linewidth=0.5)
plt.ylabel('∂(final X position) / ∂(activation)')
plt.title('Sensitivity of Final X Position to Each Muscle')
plt.xticks(rotation=45)

# Add interpretation
plt.text(0.02, 0.98, 'Blue = Increase activation → Move right\nRed = Increase activation → Move left',
         transform=plt.gca().transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Summary

In this notebook we covered:

1. **State Types**: NamedTuples for joint, muscle, geometry, and effector state
2. **State Operations**: Accessing values, creating modified states with `_replace()`
3. **Initialization**: Random vs specific initial positions
4. **Single Steps**: Using `Arm26.step()` 
5. **Trajectories**: Efficient simulation with `jax.lax.scan`
6. **Time-Varying Actions**: Passing action sequences through scan
7. **Batched Simulation**: Parallel simulation with different conditions
8. **External Loads**: Applying forces to the arm
9. **Gradients**: Computing gradients through simulations

### Key JAX Patterns:

| Pattern | Use Case |
|---------|----------|
| `jax.jit` | Compile functions for speed |
| `jax.lax.scan` | Efficient loops (trajectories) |
| `jax.vmap` | Parallel over batch dimension |
| `jax.grad` | Gradients through simulation |
| `static_argnums` | Tell JIT which args are compile-time constants |